# Attention-Sink Analysis (Qwen3-1.7B)  
### Google Colab notebook (2 of 2)

**Before running:** `Runtime > Change runtime type > Hardware accelerator > GPU` (a free **T4** is enough for Qwen3-1.7B).

To open in Colab: go to [colab.research.google.com](https://colab.research.google.com) > `File > Upload notebook`, and drop this `.ipynb` in. Run the cells top to bottom.

The first cell installs dependencies and the second mounts Google Drive so data persists between the two notebooks. Set `USE_DRIVE = False` in both if you prefer ephemeral storage.

---

In [ ]:
# --- Colab environment setup -------------------------------------------------
# Detect Colab, install the versions Qwen3 needs, and confirm a GPU is attached.
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    import subprocess, sys
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q', '-U',
         'transformers>=4.51', 'datasets>=2.19', 'accelerate', 'pyarrow'],
        check=True,
    )

import torch
print('In Colab      :', IN_COLAB)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU           :', torch.cuda.get_device_name(0))
    print('bf16 native   :', torch.cuda.is_bf16_supported(), '(T4 = False -> fp16 is used)')
else:
    print('*** No GPU. In Colab: Runtime > Change runtime type > Hardware accelerator > GPU (T4). ***')

## 0. Storage (Google Drive)

In [ ]:
# --- Storage location --------------------------------------------------------
# Point at the SAME Drive folder Notebook 1 wrote to, so a fresh Colab session
# can read the raw tensors without re-running extraction.
USE_DRIVE = True   # must match the choice made in Notebook 1

from pathlib import Path
if IN_COLAB and USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = Path('/content/drive/MyDrive/attention_sink_project')
elif IN_COLAB:
    BASE = Path('/content/attention_sink_project')
else:
    BASE = Path('.')
BASE.mkdir(parents=True, exist_ok=True)

DATA_ROOT = str(BASE / 'attention_sink_data')            # <- AnalysisConfig.data_dir
FIG_ROOT  = str(BASE / 'attention_sink_data' / 'figures')  # <- AnalysisConfig.fig_dir
print('Data root :', DATA_ROOT)
print('Fig  root :', FIG_ROOT)

# Notebook 2 — Attention-Sink Analysis (Qwen3-1.7B)

Derives every statistic from the **raw** attention tensors written by Notebook 1.
Attention sinks are treated as a **depth-** and **head-**dependent phenomenon;
nothing is collapsed to a single scalar unless explicitly as a summary.

### Methodology requirements addressed here
| # | Requirement | Where |
|---|-------------|-------|
| 1 | Depth-dependence: sink metric for **every** layer + layer-progression plot (main figure) | §6 |
| 2 | Preserve head info: **Layer × Head** sink matrix as the primary representation + derived stats | §4, §7 |
| 3 | Never average across layers as a primary analysis (only optional global summary) | §4–§6 |
| 4 | Reconsider query-position averaging (causal-mask bias); compare multiple sink definitions | §3, §15 |
| 5 | Sequence-length robustness (En vs Vi tokenise differently) | §14 |
| 8 | Publication-quality figures (curves, heatmaps, box/violin, distributions) | §6–§13 |
| 9 | Derive everything from saved raw tensors | throughout |

### Analyses implemented (per the checklist)
Layer × Head matrix · Layer progression curve (28 layers) · First/Middle/Last
qualitative heatmaps · Head heatmaps · Layer heatmaps · Statistics across
prompts · Boxplots · Violin plots · Sequence-length plots · DataFrame output.

## 1. Imports & configuration

In [ ]:
import json
from dataclasses import dataclass, asdict
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt

mpl.rcParams.update({
    'figure.dpi': 120, 'savefig.dpi': 150, 'font.size': 10,
    'axes.grid': True, 'grid.alpha': 0.25, 'axes.axisbelow': True,
})
EN_C, VI_C = '#1f77b4', '#d62728'   # consistent language colours

@dataclass
class AnalysisConfig:
    data_dir: str = DATA_ROOT
    fig_dir: str  = FIG_ROOT
    sink_threshold: float = 0.30     # a (layer,head) is a 'sink head' if score > threshold
    query_skip_k: int = 4            # drop first k query positions -> length-debiased metric
    primary_metric: str = 'mean_from_k'
    save_figs: bool = True

acfg = AnalysisConfig()
FIG = Path(acfg.fig_dir); FIG.mkdir(parents=True, exist_ok=True)

def savefig(name):
    if acfg.save_figs:
        plt.savefig(FIG / name, bbox_inches='tight')
print(json.dumps(asdict(acfg), indent=2))

## 2. Load raw attention & metadata (Requirement 9)

In [ ]:
DATA = Path(acfg.data_dir)
meta = pd.read_csv(DATA / 'metadata.csv')
with open(DATA / 'config.json') as f:
    run_cfg = json.load(f)

NUM_LAYERS = int(meta['n_layers'].iloc[0])
NUM_HEADS  = int(meta['n_heads'].iloc[0])
meta_idx = meta.set_index('prompt_id')

def load_attn(prompt_id):
    '''Load one raw attention tensor [L, H, S, S] (float32) + its token ids.'''
    d = np.load(DATA / 'attn' / (prompt_id + '.npz'))
    return d['attn'].astype(np.float32), d['token_ids']

print('prompts        :', len(meta), '| by language:', meta['language'].value_counts().to_dict())
print('layers x heads :', NUM_LAYERS, 'x', NUM_HEADS)
print('prompt_mode    :', run_cfg['prompt_mode'], '| token0 standardised:', meta['token0_id'].nunique() == 1)
print('seq_len range  :', int(meta['seq_len'].min()), '-', int(meta['seq_len'].max()))

## 3. Sink-score definitions — addressing the query-position bias (Requirement 4)

The original metric averages attention to token 0 over **every** query position.
Because of causal masking, query 0 can attend only to key 0 (weight = 1 by
construction), query 1 only to {0,1}, and so on. Early queries are therefore
**structurally** biased toward token 0, and averaging over all of them makes the
score depend on sequence length rather than on genuine sink behaviour.

We define **four** sink metrics so definitions can be compared rather than
assuming the original is optimal. Each maps a raw `[L, H, S, S]` tensor to an
`[L, H]` matrix (attention to key 0, aggregated over query positions differently):

- **`mean_all`** — original: mean over *all* queries (length-biased baseline).
- **`mean_from_k`** — mean over queries `>= k`, dropping the trivially-saturated early rows (**primary**).
- **`mean_second_half`** — mean over the later half of queries.
- **`last`** — attention from the *final* query only (single position, length-robust).

In [ ]:
def m_mean_all(A):
    return A[:, :, :, 0].mean(axis=2)                       # [L,H]

def m_mean_from_k(A, k=acfg.query_skip_k):
    s = min(k, A.shape[2] - 1)
    return A[:, :, s:, 0].mean(axis=2)

def m_mean_second_half(A):
    s = A.shape[2] // 2
    return A[:, :, s:, 0].mean(axis=2)

def m_last(A):
    return A[:, :, -1, 0]                                   # [L,H]

METRICS = {
    'mean_all':         m_mean_all,
    'mean_from_k':      m_mean_from_k,
    'mean_second_half': m_mean_second_half,
    'last':             m_last,
}
print('metrics:', list(METRICS))
assert acfg.primary_metric in METRICS

## 4. Layer × Head sink matrix — the primary representation (Requirements 2 & 3)

For every prompt and every metric we compute the full `[L, H]` matrix and keep
it. From it we derive the summary statistics the brief asks for (layer mean,
head mean, global, per-layer/per-head maxima, fraction of sink heads) **without
discarding the matrix**.

In [ ]:
# metric -> prompt_id -> [L, H] matrix (nothing averaged away)
sink_matrices = {mn: {} for mn in METRICS}
for pid in meta['prompt_id']:
    A, _ = load_attn(pid)
    for mn, fn in METRICS.items():
        sink_matrices[mn][pid] = fn(A).astype(np.float32)

def derived_stats(M, thr):
    '''Summaries derived FROM the Layer x Head matrix (matrix itself retained).'''
    return {
        'layer_mean':     M.mean(axis=1),   # [L]
        'head_mean':      M.mean(axis=0),   # [H]
        'global':         float(M.mean()),  # optional scalar summary
        'max_per_layer':  M.max(axis=1),    # [L]
        'max_per_head':   M.max(axis=0),    # [H]
        'frac_sink_heads': float((M > thr).mean()),
    }

# demo on one prompt
_ex = meta['prompt_id'].iloc[0]
_st = derived_stats(sink_matrices[acfg.primary_metric][_ex], acfg.sink_threshold)
print('example prompt        :', _ex)
print('L x H matrix shape    :', sink_matrices[acfg.primary_metric][_ex].shape)
print('global sink score     : %.3f' % _st['global'])
print('fraction of sink heads: %.3f (threshold=%.2f)' % (_st['frac_sink_heads'], acfg.sink_threshold))

## 5. Tidy DataFrame output

In [ ]:
# Long/tidy frame: one row per (prompt, metric, layer, head). The granular record.
rows = []
for mn in METRICS:
    for pid, M in sink_matrices[mn].items():
        info = meta_idx.loc[pid]
        L, H = M.shape
        for l in range(L):
            for h in range(H):
                rows.append({
                    'prompt_id': pid, 'pair_id': int(info['pair_id']),
                    'language': info['language'], 'category': info['category'],
                    'seq_len': int(info['seq_len']), 'metric': mn,
                    'layer': l, 'head': h, 'sink_score': float(M[l, h]),
                })
df = pd.DataFrame(rows)
try:
    df.to_parquet(DATA / 'sink_scores_long.parquet')
except Exception:
    pass
df.to_csv(DATA / 'sink_scores_long.csv', index=False)
print('tidy frame:', df.shape)
df.head()

## 6. Layer-progression curve — 28 layers (Requirement 1, main figure)

The headline result: sink strength as a function of layer depth, per language,
with a ±1 std band across prompts. This replaces any Layer-0-only figure.

In [ ]:
def layer_progression(metric):
    fig, ax = plt.subplots(figsize=(8.5, 5))
    x = np.arange(NUM_LAYERS)
    for lang, c in [('en', EN_C), ('vi', VI_C)]:
        pids = meta.loc[meta['language'] == lang, 'prompt_id']
        stack = np.stack([sink_matrices[metric][p].mean(axis=1) for p in pids])  # [P, L]
        mean, sd = stack.mean(0), stack.std(0)
        ax.plot(x, mean, color=c, lw=2, label='%s (n=%d)' % (lang, len(pids)))
        ax.fill_between(x, mean - sd, mean + sd, color=c, alpha=0.15)
    ax.set_xlabel('Layer depth'); ax.set_ylabel('Mean sink score (%s)' % metric)
    ax.set_title('Attention-sink strength vs layer depth  (Qwen3-1.7B, %d layers)' % NUM_LAYERS)
    ax.set_xlim(0, NUM_LAYERS - 1); ax.legend()
    savefig('fig_layer_progression.png'); plt.show()

layer_progression(acfg.primary_metric)

## 7. Layer × Head heatmaps (primary representation, visualised)

In [ ]:
def lh_heatmap(metric, lang=None, ax=None):
    pids = meta['prompt_id'] if lang is None else meta.loc[meta['language'] == lang, 'prompt_id']
    M = np.stack([sink_matrices[metric][p] for p in pids]).mean(0)   # [L, H]
    own = ax is None
    if own:
        fig, ax = plt.subplots(figsize=(6.5, 7))
    im = ax.imshow(M, aspect='auto', cmap='magma', origin='lower')
    ax.set_xlabel('Head'); ax.set_ylabel('Layer')
    ax.set_title('%s' % (lang or 'all prompts'))
    ax.grid(False)
    plt.colorbar(im, ax=ax, shrink=0.85, label='sink score')
    return M

fig, axes = plt.subplots(1, 3, figsize=(17, 6))
for ax, lang in zip(axes, [None, 'en', 'vi']):
    lh_heatmap(acfg.primary_metric, lang, ax=ax)
fig.suptitle('Layer x Head sink matrix (%s)  —  sink heads are sparse, not uniform' % acfg.primary_metric, y=1.02)
savefig('fig_layer_head_heatmap.png'); plt.show()

## 8. First / Middle / Last qualitative attention heatmaps

Qualitative examples only (per the brief), not the primary result. Column 0 is
the sink column; note how mass concentrates there in deeper layers.

In [ ]:
pid = meta.loc[meta['language'] == 'en', 'prompt_id'].iloc[0]
A, tok = load_attn(pid)
layers = [0, NUM_LAYERS // 2, NUM_LAYERS - 1]

fig, axes = plt.subplots(1, 3, figsize=(16, 5.2))
for ax, l in zip(axes, layers):
    Ah = A[l].mean(0)                       # mean over heads -> [S, S], qualitative
    im = ax.imshow(Ah, cmap='viridis', vmin=0, vmax=float(Ah.max()))
    ax.set_title('Layer %d' % l); ax.set_xlabel('Key position'); ax.set_ylabel('Query position')
    ax.grid(False)
    ax.axvline(0, color='w', lw=0.8, ls='--', alpha=0.6)   # highlight sink column
    plt.colorbar(im, ax=ax, shrink=0.8)
fig.suptitle('Attention maps (mean over heads) — prompt %s  [qualitative]' % pid, y=1.02)
savefig('fig_first_mid_last_maps.png'); plt.show()

## 9. Head-level analysis (head heatmaps / distributions)

In [ ]:
metric = acfg.primary_metric
# prompts x heads: head-mean sink score (averaged over layers) per prompt, ordered by language
order = meta.sort_values(['language', 'pair_id'])['prompt_id'].tolist()
Hmat = np.stack([sink_matrices[metric][p].mean(axis=0) for p in order])   # [P, H]

fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
im = axes[0].imshow(Hmat, aspect='auto', cmap='magma', origin='lower')
axes[0].set_xlabel('Head'); axes[0].set_ylabel('Prompt (grouped by language)')
axes[0].set_title('Per-prompt head sink profile'); axes[0].grid(False)
plt.colorbar(im, ax=axes[0], shrink=0.85, label='sink score')

# distribution of per-head max sink score across the network
head_max = np.stack([sink_matrices[metric][p].max(axis=0) for p in meta['prompt_id']]).mean(0)  # [H]
axes[1].bar(np.arange(NUM_HEADS), head_max, color='#555')
axes[1].axhline(acfg.sink_threshold, color=VI_C, ls='--', label='sink threshold')
axes[1].set_xlabel('Head'); axes[1].set_ylabel('mean over prompts of max-over-layers sink')
axes[1].set_title('Which heads specialise as sinks'); axes[1].legend()
savefig('fig_head_analysis.png'); plt.show()

## 10. Layer-level analysis (layer heatmaps)

In [ ]:
Lmat = np.stack([sink_matrices[metric][p].mean(axis=1) for p in order])   # [P, L]
fig, ax = plt.subplots(figsize=(11, 5.5))
im = ax.imshow(Lmat, aspect='auto', cmap='viridis', origin='lower')
n_en = int((meta['language'] == 'en').sum())
ax.axhline(n_en - 0.5, color='w', lw=1.2)   # en/vi divider (order groups en first)
ax.set_xlabel('Layer'); ax.set_ylabel('Prompt (en above line, vi below)')
ax.set_title('Per-prompt layer sink profile (%s)' % metric); ax.grid(False)
plt.colorbar(im, ax=ax, shrink=0.85, label='sink score')
savefig('fig_layer_heatmap.png'); plt.show()

## 11. Statistics across prompts

In [ ]:
# Per-prompt summaries derived from each Layer x Head matrix (primary metric).
per_prompt = []
for p in meta['prompt_id']:
    M = sink_matrices[acfg.primary_metric][p]
    info = meta_idx.loc[p]
    per_prompt.append({
        'prompt_id': p, 'language': info['language'], 'category': info['category'],
        'seq_len': int(info['seq_len']),
        'global': float(M.mean()),
        'max_head': float(M.max()),
        'frac_sink': float((M > acfg.sink_threshold).mean()),
        'peak_layer': int(M.mean(axis=1).argmax()),
    })
pp = pd.DataFrame(per_prompt)

print('=== by language ===')
print(pp.groupby('language')[['global', 'max_head', 'frac_sink', 'peak_layer']].agg(['mean', 'std']).round(3))
print('\n=== by category ===')
print(pp.groupby('category')[['global', 'frac_sink']].agg(['mean', 'std']).round(3))
pp.head()

## 12. Boxplots (distributions, not just means — Requirement 8)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
# global sink score by language
langs = ['en', 'vi']
axes[0].boxplot([pp.loc[pp['language'] == L, 'global'] for L in langs], labels=langs, showmeans=True)
axes[0].set_ylabel('global sink score'); axes[0].set_title('Global sink score by language')
# global sink score by category
cats = sorted(pp['category'].unique())
axes[1].boxplot([pp.loc[pp['category'] == c, 'global'] for c in cats], labels=cats, showmeans=True)
axes[1].set_ylabel('global sink score'); axes[1].set_title('Global sink score by category')
axes[1].tick_params(axis='x', rotation=20)
savefig('fig_boxplots.png'); plt.show()

## 13. Violin plots

In [ ]:
# Distribution of per-(layer,head) sink scores, by language, at three representative depths.
depths = {'first': 0, 'middle': NUM_LAYERS // 2, 'last': NUM_LAYERS - 1}
fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=True)
for ax, (name, l) in zip(axes, depths.items()):
    data, labels = [], []
    for lang in ['en', 'vi']:
        pids = meta.loc[meta['language'] == lang, 'prompt_id']
        vals = np.concatenate([sink_matrices[acfg.primary_metric][p][l] for p in pids])  # heads x prompts
        data.append(vals); labels.append(lang)
    parts = ax.violinplot(data, showmeans=True, showextrema=True)
    ax.set_xticks([1, 2]); ax.set_xticklabels(labels)
    ax.set_title('%s layer (%d)' % (name, l))
axes[0].set_ylabel('per-head sink score')
fig.suptitle('Head-level sink-score distributions by language and depth (%s)' % acfg.primary_metric, y=1.02)
savefig('fig_violins.png'); plt.show()

## 14. Sequence-length analysis (Requirement 5)

English and Vietnamese tokenise into different numbers of tokens, so a raw
language difference could really be a length effect. We (a) show the seq-length
distributions per language, and (b) plot the global sink score against sequence
length for **each** metric, with a fitted line and Pearson *r*. A length-robust
metric should show little slope; the biased `mean_all` should show the most.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
# (a) seq_len by language
axes[0].boxplot([meta.loc[meta['language'] == L, 'seq_len'] for L in ['en', 'vi']],
                labels=['en', 'vi'], showmeans=True)
axes[0].set_ylabel('sequence length (tokens)')
axes[0].set_title('Tokenised length by language (potential confound)')
# (b) tokens-per-pair ratio
piv = meta.pivot_table(index='pair_id', columns='language', values='seq_len')
ratio = (piv['vi'] / piv['en']).dropna()
axes[1].hist(ratio, bins=12, color='#555')
axes[1].axvline(1.0, color=VI_C, ls='--')
axes[1].set_xlabel('vi / en token-count ratio (per matched pair)')
axes[1].set_title('Vietnamese vs English length ratio (mean %.2f)' % ratio.mean())
axes[1].grid(True, alpha=0.25)
savefig('fig_seqlen_by_language.png'); plt.show()

In [ ]:
fig, axes = plt.subplots(1, len(METRICS), figsize=(4.4 * len(METRICS), 4.2), sharey=False)
len_corr = {}
for ax, mn in zip(axes, METRICS):
    g = np.array([sink_matrices[mn][p].mean() for p in meta['prompt_id']])
    x = meta['seq_len'].values.astype(float)
    for lang, c in [('en', EN_C), ('vi', VI_C)]:
        m = meta['language'].values == lang
        ax.scatter(x[m], g[m], s=20, alpha=0.75, color=c, label=lang)
    coef = np.polyfit(x, g, 1)
    r = float(np.corrcoef(x, g)[0, 1])
    len_corr[mn] = r
    xs = np.linspace(x.min(), x.max(), 50)
    ax.plot(xs, np.polyval(coef, xs), 'k--', lw=1)
    ax.set_title('%s\nr(len)=%.2f' % (mn, r)); ax.set_xlabel('sequence length')
axes[0].set_ylabel('global sink score'); axes[0].legend()
fig.suptitle('Sink score vs sequence length by metric  (flatter = less length-biased)', y=1.03)
savefig('fig_seqlen_vs_sink.png'); plt.show()
print('|corr with seq_len|  (lower is better):',
      {k: round(v, 3) for k, v in sorted(len_corr.items(), key=lambda kv: abs(kv[1]))})

## 15. Metric comparison (Requirement 4)

Side-by-side comparison of the four sink definitions: how much they disagree,
and how strongly each depends on sequence length. This makes explicit that the
original `mean_all` is **not** assumed optimal.

In [ ]:
summary = []
for mn in METRICS:
    g = np.array([sink_matrices[mn][p].mean() for p in meta['prompt_id']])
    summary.append({
        'metric': mn,
        'mean_global': float(g.mean()),
        'std_global': float(g.std()),
        'abs_corr_seqlen': abs(len_corr[mn]),
    })
summary = pd.DataFrame(summary).sort_values('abs_corr_seqlen')
print(summary.round(3).to_string(index=False))

# Per-layer curves for all metrics (all prompts) to show they can rank layers differently.
fig, ax = plt.subplots(figsize=(9, 5))
for mn in METRICS:
    curve = np.stack([sink_matrices[mn][p].mean(axis=1) for p in meta['prompt_id']]).mean(0)
    ax.plot(np.arange(NUM_LAYERS), curve, lw=2, label=mn)
ax.set_xlabel('Layer'); ax.set_ylabel('mean sink score'); ax.set_xlim(0, NUM_LAYERS - 1)
ax.set_title('Layer progression under each sink definition'); ax.legend()
savefig('fig_metric_comparison.png'); plt.show()

## 16. Summary & limitations

**What the pipeline produces.** A full `Layer × Head` sink matrix per prompt
(the primary record), the 28-layer progression curve as the headline figure,
qualitative first/middle/last maps, head- and layer-level heatmaps, cross-prompt
box/violin distributions, an explicit sequence-length analysis, a four-way
comparison of sink definitions, and a tidy long-format DataFrame
(`sink_scores_long.csv`).

**Interpretation notes (fill in from your run).**
- Depth: read the peak-layer and shape of the progression curve — sinks are expected to strengthen beyond Layer 0, which is why Layer 0 alone is uninformative.
- Heads: the `Layer × Head` heatmap shows whether sinks concentrate in a few specialised heads rather than spreading uniformly.
- Language: any en/vi gap in §11–§13 must be cross-checked against §14 — if it tracks the length difference, it is a tokenisation confound, not a language effect.

**Known limitations.**
- `mean_all` is length-biased by causal masking; prefer `mean_from_k` / `last` (see §14–§15). The bias is documented, not hidden.
- Attention weights are one lens on sinks; they don't capture value-vector magnitude or downstream effect on the residual stream.
- The embedded fallback corpus is small; use FLORES-200 (set in Notebook 1's config) for statistical power.
- Sink identity depends on the token-0 standardisation chosen in Notebook 1 (`prompt_mode`); results are conditional on it.

## Download results

In [ ]:
# --- Bundle figures + tidy DataFrame for download -----------------------------
import shutil
bundle = Path('/content/sink_analysis_outputs') if IN_COLAB else Path('sink_analysis_outputs')
bundle.mkdir(parents=True, exist_ok=True)
shutil.copytree(FIG_ROOT, bundle / 'figures', dirs_exist_ok=True)
for f in ['sink_scores_long.csv', 'sink_scores_long.parquet']:
    src = Path(DATA_ROOT) / f
    if src.exists():
        shutil.copy(src, bundle / f)
zp = shutil.make_archive(str(bundle), 'zip', bundle)
print('Bundled outputs:', zp)
if IN_COLAB:
    from google.colab import files
    files.download(zp)